# Customer Segmentation Analysis
**OIBSIP Data Analytics — Level 1, Task 2**

This project uses K-Means to group customers by purchase behaviour. The dataset is synthetic demo data.

## 1. Load the dataset

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

BASE = Path.cwd()
if not (BASE / "data/ecommerce_customers.csv").exists() and (BASE / "Customer_Segmentation_Analysis/data/ecommerce_customers.csv").exists():
    BASE = BASE / "Customer_Segmentation_Analysis"
df = pd.read_csv(BASE / "data/ecommerce_customers.csv")
df.head()


## 2. Data quality and descriptive statistics

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.to_frame("data_type"))
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate customer IDs:", df.CustomerID.duplicated().sum())
display(df.describe().round(2))


## 3. Select and scale features
Recency is days since the last purchase, Frequency is purchase count, and Monetary value is total spend. Scaling helps prevent larger units from dominating clustering.

In [ ]:
features = ["DaysSinceLastPurchase", "PurchaseFrequency", "TotalSpend"]
X = StandardScaler().fit_transform(df[features])


## 4. Elbow Method
The elbow plot helps compare candidate cluster counts. This project uses four clusters as a starting point; cluster quality should also be judged by interpretability.

In [ ]:
ks = list(range(2, 9))
inertias = []
for k in ks:
    model = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertias.append(model.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(ks, inertias, marker="o")
plt.title("Elbow Method for K-Means")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.show()


## 5. Fit K-Means and profile clusters

In [ ]:
df["Cluster"] = KMeans(n_clusters=4, random_state=42, n_init=10).fit_predict(X)
profile = df.groupby("Cluster").agg(
    Customers=("CustomerID", "count"),
    AvgDaysSinceLastPurchase=("DaysSinceLastPurchase", "mean"),
    AvgPurchaseFrequency=("PurchaseFrequency", "mean"),
    AvgOrderValue=("AverageOrderValue", "mean"),
    AvgTotalSpend=("TotalSpend", "mean"),
    TotalRevenue=("TotalSpend", "sum")
).round(2)
display(profile)


## 6. Visualise customer groups

In [ ]:
sns.scatterplot(data=df, x="PurchaseFrequency", y="TotalSpend", hue="Cluster", palette="tab10", s=55)
plt.title("Customer Clusters: Frequency vs Total Spend")
plt.show()
df["Cluster"].value_counts().sort_index().plot(kind="bar")
plt.title("Number of Customers per Cluster")
plt.show()


## 7. Marketing recommendations

In [ ]:
for cid, row in profile.iterrows():
    if row["AvgTotalSpend"] >= profile["AvgTotalSpend"].quantile(.75):
        action = "Reward with loyalty benefits and personalised premium offers."
    elif row["AvgDaysSinceLastPurchase"] >= profile["AvgDaysSinceLastPurchase"].median():
        action = "Try a re-engagement campaign with a relevant reminder or discount."
    elif row["AvgPurchaseFrequency"] <= profile["AvgPurchaseFrequency"].median():
        action = "Encourage repeat purchases with onboarding offers and recommendations."
    else:
        action = "Promote bundles, cross-sells and loyalty points."
    print(f"Cluster {cid}: {action}")


## 8. Conclusion
- K-Means groups customers with similar purchase behaviour.
- Cluster profiles compare recency, frequency and spending.
- Marketing actions should be tested against actual business results.

**Dataset disclosure:** This is synthetic demo data created for learning, not real customer data.